<a href="https://colab.research.google.com/github/Shahriyar799/Deep_Learning/blob/main/connecting_retriver_and_llm_RAGbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 94.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 43.6 MB/s eta 0:00:00


In [ ]:
!pip install -q langchain_huggingface
!pip install -q langchain_community
!pip install -q langchain-text-splitters
!pip install -q langchain
!pip install -q langchain_openai
!pip install langchain_chroma

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 8.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 82.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 127.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 101.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━

In [ ]:
# from dotenv import load_dotenv
# from langchain_openai import ChatOpenAI


from google.colab import userdata
from huggingface_hub import login

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import glob

In [ ]:
MODEL = "microsoft/Phi-4-mini-instruct"
DB_NAME = "vector_db"
# load_dotenv(override=True)
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
knowledge_base_path = "/content/drive/MyDrive/LLM_Engineering/week5/knowledge-base/**/*.md"

filenames = glob.glob(knowledge_base_path, recursive=True)
print(f"Found {len(filenames)} files in the knowledge base")

entire_knowledge_base = ""

for file_path in filenames:
    with open(file_path, "r", encoding="utf-8") as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"

print(f"Total characters in knowledge base: {len(entire_knowledge_base):,}")

Found 76 files in the knowledge base
Total characters in knowledge base: 304,434


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Initialize the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    is_separator_regex=False,
)

# Split the entire knowledge base into documents
documents = text_splitter.create_documents([entire_knowledge_base])

print(f"Split knowledge base into {len(documents)} documents.")
# Display the first document to verify
if documents:
    print("\nFirst document content:")
    print(documents[0].page_content)

Split knowledge base into 398 documents.

First document content:
# Careers at Insurellm

## Why Join Insurellm?

At Insurellm, we're not just building software—we're revolutionizing an entire industry. Since our founding in 2015, we've evolved from a high-growth startup to a lean, profitable company with 32 highly talented employees managing 32 active contracts across all eight of our product lines.

After reaching 200 employees in 2020, we strategically restructured in 2022-2023 to focus on sustainable growth, operational excellence, and building a world-class remote-first culture. Today, we're a tight-knit team of exceptional professionals who deliver outsized impact through automation, AI, and strategic focus on high-value enterprise clients—from regional insurers to global reinsurance partners.

### Our Culture


With the documents created, we can now add them to our `vectorstore`. This will embed the document chunks and store them, making them searchable by the `retriever`.

In [ ]:
vectorstore.add_documents(documents)

# Re-initialize the retriever to ensure it uses the updated vectorstore
retriever = vectorstore.as_retriever()

print("Documents added to the vectorstore and retriever re-initialized.")

Documents added to the vectorstore and retriever re-initialized.


In [ ]:
print("Retriever results for 'Who is Avery?':")
print(retriever.invoke("Who is Avery?"))

print("\nRetriever results for 'Who is Averi Lancaster?':")
print(retriever.invoke("Who is Averi Lancaster?"))

Retriever results for 'Who is Avery?':
[Document(id='79c1a95c-510e-4048-aff6-9f52a7dde66d', metadata={}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and adaptability throughout her career at Insure

In [ ]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

retriever = vectorstore.as_retriever()

hf_llm = HuggingFaceEndpoint(
    repo_id=MODEL,
    max_new_tokens=512,
    temperature=0.01,
    huggingfacehub_api_token=hf_token
)

llm = ChatHuggingFace(llm=hf_llm)

In [ ]:
retriever.invoke("Who is Averi?")

[Document(id='f546987d-303a-451f-be24-90334b273738', metadata={}, page_content='- **2021**: *Exceeds Expectations*  \n  Maxine spearheaded the transition to a new data warehousing solution, significantly enhancing Insurellm’s data analytics capabilities. This major achievement bolstered her reputation within the company.  \n\n- **2022**: *Outstanding*  \n  Maxine continued her upward trajectory, successfully implementing machine learning algorithms to predict customer behavior, which was well-received by the leadership team and improved client satisfaction.  \n\n- **2023**: *Exceeds Expectations*  \n  Maxine has taken on mentoring responsibilities and is leading a cross-functional team for data governance initiatives, showcasing her leadership and solidifying her role at Insurellm.'),
 Document(id='dc1d3ad7-9809-4e6c-ba2d-81aac564b56a', metadata={}, page_content='- **January 2021 - Present**: **Senior Data Engineer**  \n  * Maxine was promoted to Senior Data Engineer after successfully

In [ ]:
llm.invoke("Who is Avery?")

Our latest automated health check on model 'microsoft/Phi-4-mini-instruct' for provider 'featherless-ai' did not complete successfully.  Inference call might fail.


AIMessage(content='Avery is a unisex given name that has become increasingly popular in recent years. It can be used for any gender and is often chosen for its modern and versatile appeal. Avery can refer to a person, character, or even a brand, depending on the context. If you are asking about a specific Avery, such as a public figure, fictional character, or notable person, please provide more details so I can give you a more accurate answer.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 90, 'prompt_tokens': 7, 'total_tokens': 97}, 'model_name': 'microsoft/Phi-4-mini-instruct', 'system_fingerprint': 'fp1-rss-g2', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019eabfe-54d1-78c3-936b-a6b4ad546a42-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 7, 'output_tokens': 90, 'total_tokens': 97})

## Time to put this together!

In [ ]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [ ]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [ ]:
answer_question("Who is Averi Lancaster?", [])

Our latest automated health check on model 'microsoft/Phi-4-mini-instruct' for provider 'featherless-ai' did not complete successfully.  Inference call might fail.


'Avery Lancaster is the Co-Founder & Chief Executive Officer (CEO) of Insurellm. She has been instrumental in guiding the company to become a leading Insurance Tech provider. Avery is recognized for her innovative leadership strategies and expertise in risk management, which have helped Insurellm gain a strong foothold in the mainstream insurance market. Avery Lancaster was born on March 15, 1985, and is based in San Francisco, California, with a current salary of $225,000.'

In [ ]:
gr.ChatInterface(answer_question, ).launch()

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ad654124d2a1f9f6b3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Admit it - you thought RAG would be more complicated than that!!